[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/04_grounded_data_augmentation.ipynb)

# Step 4 — Retrieval-Augmented Synthetic Data

Generate **grounded** Q&A from TRAIN paragraphs to build the SFT corpus (target: 500–1,000 samples).

## Learning objectives
- Retrieve relevant passages from policy documents
- Generate faithful Q&A pairs with a teacher LLM
- Verify grounding with lexical overlap or embedding similarity

Step 2 data teaches how to prompt; Step 4 data is what you train on
Grounded generation + overlap check reduces hallucinated training labels
In a real deployment you'd have more documents and actually hit 500–1000; the bootcamp simulates the pipeline at small scale


In [2]:
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_SYNTHETIC_TARGET_SIZE,
    PARAGRAPHS_PATH,
    SYNTHETIC_TRAIN_PATH,
    Paragraph,
    ParagraphSplit,
    QASample,
    apply_heuristic_filters,
    create_teacher_client,
    effective_synthetic_target,
    generate_grounded_training_corpus,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from dotenv import load_dotenv


load_dotenv()
use_repo_root(Path("."))

2026-06-11 16:30:29,728 INFO root: AI Engineering synthetic data utilities 

 Logging configured.


PosixPath('/Users/royajavadi/projects/synthetic-data-bootcamp')

In [1]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Load TRAIN paragraphs (exclude test holdout)

In [3]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
print(f"Train paragraphs: {len(train_paragraphs)}")

Train paragraphs: 7


## 2. Generate grounded Q&A from retrieved passages

In [4]:
teacher = create_teacher_client()
target_size = effective_synthetic_target(
    train_paragraphs,
    requested=DEFAULT_SYNTHETIC_TARGET_SIZE,
)
print(f"Grounded generation target: {target_size}")

grounded_candidates = generate_grounded_training_corpus(
    teacher,
    train_paragraphs,
    target_size=target_size,
    min_overlap=0.15,
)
print(f"Generated {len(grounded_candidates)} grounded candidates")
grounded_candidates[0].metadata

Grounded generation target: 28
Generated 28 grounded candidates


{'generation_strategy': 'grounded_rag', 'grounding_overlap': 0.875}

## 3. Filter and save final SFT corpus

In [5]:
final_samples, rejected = apply_heuristic_filters(grounded_candidates)
print(f"Final SFT corpus size: {len(final_samples)} (rejected {len(rejected)})")

save_typed_jsonl(
    SYNTHETIC_TRAIN_PATH,
    final_samples,
    to_dict=QASample.to_dict,
)
SYNTHETIC_TRAIN_PATH

Final SFT corpus size: 7 (rejected 21)


PosixPath('implementations/qa_text_generation/data/synthetic/synthetic_train.jsonl')